# 04 Gold - Retail

**Audience:** participants learning the AIDP medallion pattern with PySpark.

**Prerequisites:** use the lab's shared compute and run the previous notebook first.

**Learning goal:** Builds the industry KPIs from accepted Silver records.

## Outline

1. Inspect the participant-scoped inputs.
2. Transform and persist this medallion layer.
3. Register external tables when this layer owns them.
4. Verify the row counts printed by the final statements.


In [ ]:
import re
# oidlUtils is injected by AIDP Workbench; it is not an importable module.

def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name)
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")

if re.fullmatch(r"u_[0-9a-f]{16}", participant_key) is None:
    raise ValueError("Invalid participant_key")
if lab_id != 'retail':
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")

from pyspark.sql import functions as F

silver = {"customers": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/retail/customers/", "order_items": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/retail/order_items/", "orders": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/retail/orders/", "products": f"oci://{bucket_name}@{objectstorage_namespace}/03_silver/users/{participant_key}/retail/products/"}
gold = {"customer_value": f"oci://{bucket_name}@{objectstorage_namespace}/04_gold/users/{participant_key}/retail/retail_customer_value/", "product_daily": f"oci://{bucket_name}@{objectstorage_namespace}/04_gold/users/{participant_key}/retail/retail_product_daily/"}
customers = spark.read.format("delta").load(silver["customers"])
products = spark.read.format("delta").load(silver["products"])
orders = spark.read.format("delta").load(silver["orders"])
items = spark.read.format("delta").load(silver["order_items"])
lines = (items
    .join(
        orders.select("participant_key", "order_id", "customer_id", "order_time", "order_status"),
        ["participant_key", "order_id"],
    )
    .join(products.select("participant_key", "product_id", "unit_cost"), ["participant_key", "product_id"])
    .withColumn("gross", F.col("quantity") * F.col("unit_price"))
    .withColumn("net", F.col("gross") - F.col("discount_amount")))
customer_value = (lines.groupBy("participant_key", "customer_id")
    .agg(F.countDistinct("order_id").alias("order_count"), F.sum("quantity").alias("units"), F.sum("gross").alias("gross_revenue"), F.sum("discount_amount").alias("discount_amount"), F.sum("net").alias("net_revenue"), F.max("order_time").alias("last_order_at"))
    .withColumn("average_order_value", F.round(F.col("net_revenue") / F.col("order_count"), 2))
    .select("participant_key", "customer_id", "order_count", "units", "gross_revenue", "discount_amount", "net_revenue", "average_order_value", "last_order_at"))
customer_value.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold["customer_value"])
product_daily = (lines.withColumn("order_date", F.to_date("order_time"))
    .groupBy("participant_key", "order_date", "product_id")
    .agg(F.sum("quantity").alias("units"), F.sum("net").alias("net_revenue"), F.sum(F.col("net") - F.col("quantity") * F.col("unit_cost")).alias("gross_margin"), F.sum(F.when(F.col("order_status") == "refunded", F.col("quantity")).otherwise(0)).alias("refunded_units")))
product_daily.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(gold["product_daily"])
customer_value.show(20, truncate=False)

for table_name, location in gold.items():
    row_count = spark.read.format("delta").load(location).count()
    assert row_count > 0, f"Gold table {table_name} is empty"
    print(f"Gold {table_name}: {row_count} rows")

spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_gold.{participant_key}_retail_customer_value (`participant_key` STRING, `customer_id` STRING, `order_count` BIGINT, `units` BIGINT, `gross_revenue` DOUBLE, `discount_amount` DOUBLE, `net_revenue` DOUBLE, `average_order_value` DOUBLE, `last_order_at` TIMESTAMP) USING DELTA LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/04_gold/users/{participant_key}/retail/retail_customer_value/'""")
spark.sql(f"""CREATE EXTERNAL TABLE IF NOT EXISTS aidp_lab.oci_gold.{participant_key}_retail_product_daily (`participant_key` STRING, `order_date` DATE, `product_id` STRING, `units` BIGINT, `net_revenue` DOUBLE, `gross_margin` DOUBLE, `refunded_units` BIGINT) USING DELTA LOCATION 'oci://{bucket_name}@{objectstorage_namespace}/04_gold/users/{participant_key}/retail/retail_product_daily/'""")


## Expected result

Two non-empty, industry-specific aggregate Delta tables are registered.

**Exercise:** rerun this notebook and confirm that counts do not increase. All writes use
participant-exclusive paths and overwrite mode, so a second run is idempotent.

**Common pitfall:** do not replace the participant paths with shared locations. That would mix
different students' data. As an extension, query the registered tables with `spark.sql`.
